# The shift baseline — a real submission for **T1, T2 and T3** in ~10 lines each

[`veckit`](https://github.com/aristoteleo/veckit) *scores* a submission. This notebook *makes* one, for
every task in the Virtual Embryo Challenge.

`pseudobulk_shift` is one of the two reference baselines shipped with the challenge, and it is the one
worth understanding first: the simplest method that is actually about **change** rather than about
copying. No training, no GPU, runs in a second.

**One idea, three tasks.** Measure how a population moved between two things you *can* observe, then add
that movement to the real cells you are asked to predict from. What differs is only *what the two things
are*:

| task | measure the shift between | apply it to |
|---|---|---|
| **T1** temporal | the two most recent **stages** | the last stage's cells |
| **T2** spatial-temporal | the two most recent **stages** | the last stage's cells + carry their 3D coordinates |
| **T3** perturbation | the training **knockout** and its wild type | the target's wild-type cells |

$$\Delta = \overline{X}^{\text{after}} - \overline{X}^{\text{before}}, \qquad
  \widehat x_i = \max\bigl(0,\ x_i + \Delta\bigr)$$

For T1/T2 the mean is taken **per cell type** ($\Delta_k$, applied to cells of type $k$), because cells of
the same type drift together; $\Delta_k = 0$ for a type with no counterpart in the earlier stage — no
observed transition means nothing to estimate, so those cells stay put. For T3 it is a single global
$\Delta$, since a knockout's effect is what you are transferring, not a per-type developmental drift.

Two properties make this a strong floor rather than a toy:

* it moves the **pseudobulk** by the observed amount, so it gets the *direction* of change right;
* it keeps the **real cells** — adding a constant preserves within-cell-type heterogeneity, instead of
  collapsing every cell onto a fitted mean the way a "predict the average" model would.

## Setup

`veckit` for scoring, plus the same 150-cell sample stages its own tutorial uses. Swap these two paths for
the released T1 stages ([virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data)) and
nothing else in this notebook changes.

In [ ]:
!pip install -q veckit

!mkdir -p sample_data
!wget -q -O sample_data/T1_8.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_8.5.h5ad
!wget -q -O sample_data/T1_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_9.5.h5ad
!wget -q -O sample_data/T2_9.25.h5ad https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_heart_9.25.h5ad
!wget -q -O sample_data/T2_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_heart_9.5.h5ad
!wget -q -O sample_data/T3_wt.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_wt.h5ad
!wget -q -O sample_data/T3_ko.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_mab21l2_ko.h5ad


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Task 1 — temporal

Δ per cell type between the two most recent stages, added to the last stage's real cells.

Two conventions that matter and are easy to get wrong:

* **`.copy()` the last stage** — the prediction is the real cells, shifted; not a fresh array of means.
  (It also carries `obs`/`obsm` through, which is what makes the same function work for T2 below.)
* **types missing from the earlier stage keep `Δ = 0`** — a population that is new at $S-1$ has no
  observed transition, so there is nothing to extrapolate and it stays where it is.

In [ ]:
import numpy as np, anndata as ad
from scipy import sparse

def dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)

def pseudobulk_shift(prev_path, last_path, celltype_key="celltype", damp=1.0):
    """T1/T2: Delta_k = mean_k(S-1) - mean_k(S-2);  x_hat_i = max(0, x_i + damp * Delta_{y_i})."""
    prev, last = ad.read_h5ad(prev_path), ad.read_h5ad(last_path)
    genes = [g for g in last.var_names if g in set(prev.var_names)]   # align the panels
    prev, last = prev[:, genes], last[:, genes]

    Xp, Xl = dense(prev.X), dense(last.X)
    cp = np.asarray(prev.obs[celltype_key]).astype(str)
    cl = np.asarray(last.obs[celltype_key]).astype(str)

    mean_prev = {c: Xp[cp == c].mean(0) for c in np.unique(cp)}
    pred = Xl.copy()                                   # start from the REAL cells
    shifted = 0
    for c in np.unique(cl):
        if c in mean_prev:                             # else Delta = 0: no observed transition
            delta = Xl[cl == c].mean(0) - mean_prev[c]
            pred[cl == c] = pred[cl == c] + damp * delta
            shifted += int((cl == c).sum())

    out = last.copy()                                  # keeps obs/obsm -> T2 gets spatial_3D for free
    out.X = np.clip(pred, 0, None).astype(np.float32)  # expression cannot go negative
    return out, shifted

pred_t1, n_shifted = pseudobulk_shift("sample_data/T1_8.5.h5ad", "sample_data/T1_9.5.h5ad")
pred_t1.write("t1_prediction.h5ad")

print(f"T1: predicted {pred_t1.n_obs} cells x {pred_t1.n_vars} genes")
print(f"    {n_shifted}/{pred_t1.n_obs} cells got a non-zero shift "
      f"({pred_t1.n_obs - n_shifted} are in types with no counterpart in the earlier stage)")

T1: predicted 150 cells x 32285 genes
    75/150 cells got a non-zero shift (75 are in types with no counterpart in the earlier stage)


## Did it actually move anything?

Before scoring, a sanity check worth doing on any submission: compare it to the stage you started from.
If the pseudobulk barely moved, the method silently degenerated into `copy_last` and no metric will tell
you that in a way you'd notice.

In [ ]:
last = ad.read_h5ad("sample_data/T1_9.5.h5ad")[:, list(pred_t1.var_names)]
X_last, X_pred = dense(last.X), dense(pred_t1.X)

pb_last, pb_pred = X_last.mean(0), X_pred.mean(0)
moved = np.abs(pb_pred - pb_last)

print(f"pseudobulk shift   mean |delta| = {moved.mean():.4f}   max = {moved.max():.4f}")
print(f"genes moved by >0.1: {int((moved > 0.1).sum())} / {len(moved)}")
print(f"corr(prediction, starting stage) across genes = {np.corrcoef(pb_pred, pb_last)[0,1]:.4f}")

# within-type spread must survive: that is the whole point of shifting real cells
print(f"\nper-gene std  starting stage = {X_last.std(0).mean():.4f}"
      f"   prediction = {X_pred.std(0).mean():.4f}   (a mean-collapse model would drop this to ~0)")

pseudobulk shift   mean |delta| = 0.0118   max = 0.3766
genes moved by >0.1: 340 / 32285
corr(prediction, starting stage) across genes = 0.9971

per-gene std  starting stage = 0.1455   prediction = 0.1535   (a mean-collapse model would drop this to ~0)


## Score it

`pseudobulk_shift_prediction.h5ad` is a submission file — hand it to `veckit` exactly as you would your
own model's output.

> **About the numbers below.** The sample data ships two stages (E8.5, E9.5) and no *third*, held-out one,
> so there is no ground truth here for what comes after E9.5. `--target` is pointed at E9.5 purely so the
> command runs end to end: it measures how far the shift moved **away from the starting stage**, not
> accuracy. On the released data you point `--target` at the true held-out stage and these become real
> scores. Everything above this cell is the genuine method, unchanged.

In [ ]:
!veckit --task T1 \
  --input t1_prediction.h5ad \
  --target sample_data/T1_9.5.h5ad \
  --reference sample_data/T1_8.5.h5ad

{
  "meta": {
    "task": "T1",
    "target_source": "sample_data/T1_9.5.h5ad",
    "reference_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 32285
  },
  "metrics": {
    "de_score": 0.5455,
    "de_direction": 0.8618,
    "energy_distance": -0.37365,
    "mmd_u": -0.00443,
    "variogram": 0.002013,
    "pb_rel_err": 0.0791,
    "library_size_ratio": 1.048,
    "variance_ratio": 1.119,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 0.9971,
    "_de_raw": 0.5455,
    "_de_chance": 0.0,
    "_de_chance_unif": 0.001,
    "_n_up": 18,
    "_n_dn": 15
  }
}


## The baseline you have to beat

`copy_last` — submit the most recent stage unchanged. It is the challenge's **floor**: the skill scale is
built so that `copy_last` sits at exactly 50 and the attainable oracle at 100. Scoring it the same way
shows what "doing nothing" looks like next to the shift.

This is also the honest reason `pseudobulk_shift` is worth knowing: on the real data it is *not* free to
beat. `copy_last` submits genuine single cells with genuine gene-gene structure, so it is very hard to
beat on distributional metrics — the shift wins on the *expression-change* metrics (`de_score`,
`de_direction`) by actually moving in the right direction.

In [ ]:
!veckit --task T1 \
  --input sample_data/T1_9.5.h5ad \
  --target sample_data/T1_9.5.h5ad \
  --reference sample_data/T1_8.5.h5ad

{
  "meta": {
    "task": "T1",
    "target_source": "sample_data/T1_9.5.h5ad",
    "reference_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 32285
  },
  "metrics": {
    "de_score": 0.5152,
    "de_direction": 1.0,
    "energy_distance": -0.8259,
    "mmd_u": -0.00813,
    "variogram": 0.0,
    "pb_rel_err": 0.0,
    "library_size_ratio": 1.0,
    "variance_ratio": 1.0,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 1.0,
    "_de_raw": 0.5152,
    "_de_chance": 0.0,
    "_de_chance_unif": 0.001,
    "_n_up": 18,
    "_n_dn": 15
  }
}


## Task 2 — spatial-temporal

T2 asks for expression **and** 3D position. The expression half is the same function, unchanged — and
because it builds the prediction with `last.copy()`, `obsm['spatial_3D']` comes along for free.

That default is the baseline's honest position on geometry: **carry the last stage's coordinates
forward**. It is a transparent reference, not a prediction — a real embryo also grows and deforms between
stages, so this is the most obvious thing in the whole notebook to improve on.

In [ ]:
pred_t2, n_shifted_t2 = pseudobulk_shift("sample_data/T2_9.25.h5ad", "sample_data/T2_9.5.h5ad")
pred_t2.write("t2_prediction.h5ad")

print(f"T2: predicted {pred_t2.n_obs} cells x {pred_t2.n_vars} genes, {n_shifted_t2} shifted")
print(f"    spatial_3D present: {'spatial_3D' in pred_t2.obsm}  shape {pred_t2.obsm['spatial_3D'].shape}")

# confirm the geometry really is carried through unchanged, not silently dropped or reordered
src = ad.read_h5ad("sample_data/T2_9.5.h5ad")
same = np.allclose(np.asarray(pred_t2.obsm["spatial_3D"], dtype=float),
                   np.asarray(src.obsm["spatial_3D"], dtype=float))
print(f"    coordinates identical to the last observed stage: {same}  (that IS the baseline's geometry)")

T2: predicted 150 cells x 500 genes, 13 shifted
    spatial_3D present: True  shape (150, 3)
    coordinates identical to the last observed stage: True  (that IS the baseline's geometry)


In [ ]:
!veckit --task T2 --setting heart \
  --input t2_prediction.h5ad \
  --target sample_data/T2_9.5.h5ad \
  --reference sample_data/T2_9.25.h5ad

{
  "meta": {
    "task": "T2",
    "setting": "heart",
    "target_source": "sample_data/T2_9.5.h5ad",
    "reference_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 500
  },
  "metrics": {
    "de_score": 0.5102,
    "de_direction": 0.9926,
    "energy_distance": -0.45518,
    "mmd_u": -0.00694,
    "variogram": 0.001607,
    "d2_shape": 0.00126,
    "sliced_wasserstein": 0.0,
    "occupancy_dice": 1.0,
    "scale_log_ratio": 0.0,
    "count_log_ratio": 0.0,
    "neighborhood_mmd": 0.00122,
    "pb_rel_err": 0.0848,
    "library_size_ratio": 1.0,
    "variance_ratio": 1.093,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 0.9983,
    "morans_I_agreement": 0.9827,
    "_de_raw": 0.7209,
    "_de_chance": 0.4302,
    "_n_up": 86,
    "_n_dn": 0,
    "_sw_flip_spread": 0.08091,
    "_dice_voxel_over_nn": 1.9
  }
}


## Task 3 — perturbation

Same idea, different pair. Here the shift is not developmental drift but a **knockout effect**: measure
it on the one knockout you are given, then transfer it onto a *different* knockout's wild type.

$$\Delta = \overline{X}^{\text{KO}_{\text{train}}} - \overline{X}^{\text{WT}_{\text{train}}}, \qquad
  \widehat x_i = \max\bigl(0,\ x_i^{\text{WT}_{\text{target}}} + \Delta\bigr)$$

One global Δ, not per cell type — you are transferring the perturbation's signature, not a per-type drift.

**This is the assumption the task exists to test**, and it is worth being explicit that it is a strong
one: it says different knockouts produce the *same* transcriptional response. They do not. That is why a
one-line transfer is the baseline rather than the answer, and why beating it is the interesting part.

An optional refinement that costs one line: also zero the knocked-out gene itself, the one effect you
know with certainty for any knockout.

In [ ]:
def shift_transfer(train_wt_path, train_ko_path, target_wt_path, ko_gene=None, damp=1.0):
    """T3: Delta = pseudobulk(KO_train) - pseudobulk(WT_train), added to the target's wild-type cells."""
    wt_tr, ko_tr = ad.read_h5ad(train_wt_path), ad.read_h5ad(train_ko_path)
    tgt = ad.read_h5ad(target_wt_path)
    genes = [g for g in tgt.var_names if g in set(wt_tr.var_names) and g in set(ko_tr.var_names)]
    wt_tr, ko_tr, tgt = wt_tr[:, genes], ko_tr[:, genes], tgt[:, genes]

    delta = dense(ko_tr.X).mean(0) - dense(wt_tr.X).mean(0)     # the knockout's signature
    pred = dense(tgt.X) + damp * delta

    out = tgt.copy()
    out.X = np.clip(pred, 0, None).astype(np.float32)
    if ko_gene is not None:                                     # the one effect known with certainty
        lower = {g.lower(): i for i, g in enumerate(out.var_names)}
        if ko_gene.lower() in lower:
            X = dense(out.X); X[:, lower[ko_gene.lower()]] = 0.0
            out.X = X.astype(np.float32)
    return out, delta

# The sample data ships ONE wild-type/knockout pair, so here the shift is measured and applied on the
# same pair. On the real task you measure it on the TRAINING knockout and apply it to a DIFFERENT one --
# that generalisation gap is the whole difficulty, and it is not visible in this cell.
pred_t3, delta = shift_transfer("sample_data/T3_wt.h5ad", "sample_data/T3_ko.h5ad",
                                "sample_data/T3_wt.h5ad", ko_gene="mab21l2")
pred_t3.write("t3_prediction.h5ad")

print(f"T3: predicted {pred_t3.n_obs} cells x {pred_t3.n_vars} genes")
print(f"    knockout signature: mean |delta| = {np.abs(delta).mean():.4f}, "
      f"{int((np.abs(delta) > 0.1).sum())} genes moved by >0.1")

T3: predicted 150 cells x 500 genes
    knockout signature: mean |delta| = 0.1876, 235 genes moved by >0.1


In [ ]:
!veckit --task T3 \
  --input t3_prediction.h5ad \
  --target sample_data/T3_ko.h5ad \
  --wt sample_data/T3_wt.h5ad

{
  "meta": {
    "task": "T3",
    "target_source": "sample_data/T3_ko.h5ad",
    "wt_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 500
  },
  "metrics": {
    "de_score": 0.641,
    "de_direction": 0.9984,
    "severity_slope": -0.0941,
    "energy_distance": -0.04947,
    "mmd_u": 0.03863,
    "variogram": 0.0664,
    "pb_rel_err": 0.1203,
    "library_size_ratio": 1.121,
    "variance_ratio": 0.874,
    "composition_JSD": 0.0076,
    "pseudobulk_pearson": 0.9915,
    "_de_raw": 0.6818,
    "_de_chance": 0.1136,
    "_n_up": 35,
    "_n_dn": 9,
    "_slope_r2": 0.886,
    "d2_shape": 0.01918,
    "sliced_wasserstein": 0.06181,
    "occupancy_dice": 0.5263,
    "scale_log_ratio": 0.1045,
    "count_log_ratio": 0.0,
    "neighborhood_mmd": 0.07887,
    "_sw_flip_spread": 0.00852,
    "_dice_voxel_over_nn": 2.0
  }
}


## Using it for real, and where to go next

Point the same two functions at the released data ([virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data)),
and `--target` at the stage/knockout you are actually predicting:

```python
# T1 / T2 — the two most recent stages you can see
pred, _ = pseudobulk_shift("E9.25.h5ad", "E9.5.h5ad")
pred.write("my_submission.h5ad")

# T3 — measure on the TRAINING knockout, apply to the TARGET's wild type
pred, _ = shift_transfer("WT_E9.5.h5ad", "mab21l2_KO.h5ad", "WT_E8.75.h5ad", ko_gene="gata4")
pred.write("my_submission.h5ad")
```
```bash
veckit --task T1 --input my_submission.h5ad --target E10.5.h5ad     --reference E9.5.h5ad
veckit --task T2 --input my_submission.h5ad --target E10.5.h5ad     --reference E9.5.h5ad --setting heart
veckit --task T3 --input my_submission.h5ad --target gata4_KO.h5ad  --wt WT_E8.75.h5ad
```

**Knobs worth turning**, in rough order of how much they matter:

* **`damp`** — scale the shift. `damp=1` assumes the next change matches the last one; if your stages are
  unevenly spaced in time they do not, and `damp = Δt_target / Δt_observed` is the first correction to try.
* **finer cell types** (T1/T2) — Δ is only as sharp as the labels it is averaged over.
* **predict the geometry** (T2) — the baseline carries coordinates forward unchanged, so anything that
  models growth and deformation is a clear gain.
* **stop assuming knockouts are interchangeable** (T3) — a per-gene response, e.g. conditioned on a
  regulatory network, is where the transfer assumption breaks and the real modelling starts.
* **stop averaging** (all tasks) — a per-cell velocity instead of a per-type constant is where the ODE
  baselines (`neural_ode`, `dynode_flow`) begin.

Scoring details and the Python API: [`veckit`'s own tutorial](veckit_tutorial.ipynb).